In [ ]:
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error  # ?? ?? ?? ??
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from sklearn.preprocessing import StandardScaler  # ?? 0, ???? 1
from sklearn.preprocessing import MinMaxScaler  # ???
from sklearn.preprocessing import RobustScaler

In [ ]:
file_url = (
    "https://media.githubusercontent.com/media/musthave-ML10/data_source/main/fraud.csv"
)

data = pd.read_csv(file_url)
print(data.shape)

In [ ]:
data


In [ ]:
data.drop(["first","last","street","city","state","zip","trans_num","unix_time","job","merchant"],axis=1,inplace=True)

In [ ]:
data.info()

In [ ]:
data["trans_date_trans_time"] = pd.to_datetime(data["trans_date_trans_time"])

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
amt_info = data.groupby("cc_num")["amt"].agg(["mean", "std"]).reset_index()

In [ ]:
amt_info

In [ ]:
data = data.merge(amt_info,on="cc_num",how="left")
data.head()

In [ ]:
data["amt_z_score"] = (data["amt"] - data["mean"]) / data["std"]

In [ ]:
data[["amt","amt_z_score","mean","std"]].head()

In [ ]:
data.drop(["mean","std"],axis=1,inplace=True)

In [ ]:
data.head()

In [ ]:
category_info = data.groupby(["cc_num","category"])["amt"].agg(["mean","std"]).reset_index()


In [ ]:
category_info.head(2)

In [ ]:
data = data.merge(category_info,on=["cc_num","category"],how="left")

In [ ]:
data.head(2)

In [ ]:
data["cat_z_score"] = (data["amt"]-data["mean"])/data["std"]

In [ ]:
data.head(2)

In [ ]:
data.drop(["mean","std"],axis=1,inplace=True)

In [ ]:
data.head(1)

In [ ]:
!uv add geopy

In [ ]:
import geopy.distance

In [ ]:
zip(data["merch_lat"],data["merch_long"])

In [ ]:
data["merch_coord"] = pd.Series(zip(data["merch_lat"],data["merch_long"]))
data["cust_coord"] = pd.Series(zip(data["lat"],data["long"]))

In [ ]:
data[["merch_coord","cust_coord"]]

In [ ]:
geopy.distance.distance(data['merch_coord'][0],data["cust_coord"][0])

In [ ]:
type(data["merch_coord"][0])

In [ ]:
data["distance"] = data.apply(
    lambda x:geopy.distance.distance(x["merch_coord"],x["cust_coord"]).km,
    axis=1
)

In [ ]:
data[data["is_fraud"]==1][["distance","is_fraud"]]

In [ ]:

data.distance.describe()

In [ ]:
distance_inf = data.groupby("cc_num")["distance"].agg(['mean',"std"]).reset_index()
data =data.merge(distance_inf,on = "cc_num",how="left")
data["distance_z_score"] = (data["distance"]-data["mean"]) / data["std"]
data.drop(["mean","std"],axis=1 , inplace=True)

In [ ]:
data.head()

In [ ]:
data[["amt","amt_z_score","cat_z_score","distance","distance_z_score"]].describe()

In [ ]:
data.trans_date_trans_time.max()

In [ ]:
pd.to_datetime(data["dob"])


In [ ]:
data["age"] = 2021 - pd.to_datetime(data["dob"]).dt.year

In [ ]:
data["age"]

In [ ]:
data["age"].value_counts()

In [ ]:
data.head(2)

In [ ]:
data.columns

In [ ]:
data = data[
    [
        "trans_date_trans_time",
        "category",
        "amt",
        "gender",
        "city_pop",
        "is_fraud",
        "amt_z_score",
        "cat_z_score",
        "distance",
        "distance_z_score",
        "age"
    ]].copy()


In [ ]:
data.info()

In [ ]:
data = pd.get_dummies(data,columns=["category","gender"],drop_first=True)

In [ ]:
data.info()

In [ ]:
data.set_index("trans_date_trans_time",inplace=True)
data.head()

In [ ]:
data.drop("dob",axis=1,inplace=True)

In [ ]:
data.index.max(),data.index.min()

In [ ]:
train = data[data.index< "2020-07-01"]
test  = data[data.index>="2020-07-01"]



In [ ]:
X_train=train.drop("is_fraud",axis=1)
X_test = test.drop("is_fraud",axis=1)
y_train = train["is_fraud"]
y_test = test["is_fraud"]

In [ ]:
model_1 = lgb.LGBMClassifier(random_state=10)
model_1.fit(X_train,y_train)
pred_1 = model_1.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(pred_1,y_test))

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc_score(y_test,pred_1)

In [ ]:
model_1.train

In [ ]:
train = data[data.index < "2020-01-01"]
val = data[ (data.index >= "2020-01-01") & (data.index < "2020-07-01")]
test = data[data.index >= "2020-07-01"]

In [ ]:
X_train = train.drop("is_fraud", axis=1)
X_val = val.drop("is_fraud", axis=1)
X_test = test.drop("is_fraud", axis=1)
y_train = train["is_fraud"]
y_val = val["is_fraud"]
y_test = test["is_fraud"]

In [ ]:
d_train = lgb.Dataset(X_train,label=y_train)
d_val = lgb.Dataset(X_val,label = y_val)


In [ ]:
params_set = {
        "lambda_l1": 0,
        "lambda_l2": 20,
        "learning_rate": 0.05,
        "max_depth": 15,
        "metric": "auc",
        "n_estimators": 1000,
        "subsample": 1,
    }


In [ ]:
model_2 =  lgb.train(params_set,d_train,valid_sets=[d_val],callbacks=[lgb.early_stopping(stopping_rounds=50),lgb.log_evaluation(50)])

In [ ]:
pred_2 = model_2.predict(X_test)

In [ ]:
print(classification_report(pred_2, y_test))

In [ ]:
roc_auc_score(y_test, pred_2)

In [ ]:
data.columns

In [ ]:
f_imp = pd.DataFrame(sorted(zip(model_2.feature_importance(),X_train.columns)),columns=["Value","Feature"])

In [ ]:
f_imp.head()

In [ ]:
plt.figure(figsize=(20,10))
sns.barplot(x="Value", y="Feature", data=f_imp.tail(10))
plt.show()

In [ ]:
[f_imp["Feature"].tail(7).values]

In [ ]:
data2 = data[
    [
        "distance_z_score",
        "distance" ,
        "amt_z_score",
        "city_pop",
        "cat_z_score",
        "age",
        "amt",
        "is_fraud",
    ]
]
data2.head(2)

In [ ]:
train = data2[data2.index < "2020-01-01"]
val = data2[(data2.index >= "2020-01-01") & (data2.index < "2020-07-01")]
test = data2[data2.index >= "2020-07-01"]

X_train = train.drop("is_fraud", axis=1)
X_val = val.drop("is_fraud", axis=1)
X_test = test.drop("is_fraud", axis=1)
y_train = train["is_fraud"]
y_val = val["is_fraud"]
y_test = test["is_fraud"]

In [ ]:
d_train = lgb.Dataset(X_train, label=y_train)
d_val = lgb.Dataset(X_val, label=y_val)

In [ ]:
params_set = {
    "lambda_l1": 0,
    "lambda_l2": 20,
    "learning_rate": 0.05,
    "max_depth": 15,
    "metric": "auc",
    "n_estimators": 1000,
    "subsample": 1,
}

In [ ]:
model_3 = lgb.train(
    params_set,
    d_train,
    valid_sets=[d_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)],
)

In [ ]:
pred_3 = model_3.predict(X_test)

In [ ]:
roc_auc_score(y_test,pred_3)